In [1]:
import pandas as pd

df = pd.read_excel("15000politician_statements.xlsx")

qualifier_dict = {}  

for _, row in df.iterrows():
    for i in range(1, 22):
        qid    = row.get(f'Qualifier_ID({i})',    '')
        qlabel = row.get(f'Qualifier_Label({i})', '')
        if not isinstance(qid, str) or qid.strip() == '':
            break
        qualifier_dict[qid.strip()] = qlabel  

result_df = pd.DataFrame(list(qualifier_dict.items()), 
                         columns=['Qualifier_ID', 'Qualifier_Label'])
result_df = result_df.sort_values('Qualifier_ID').reset_index(drop=True)

print(f"The number of unique qualifier types: {len(result_df)}")
result_df.to_excel("15000politician_qualifier_list.xlsx", index=False)
print("Saved to 15000politician_qualifier_list.xlsx")

The number of unique qualifier types: 238
Saved to 15000politician_qualifier_list.xlsx


In [2]:
import requests
import time

headers = {
    'User-Agent': 'WikidataResearch/1.0 (academic research; your_email@example.com)'
}

def get_descriptions_batch(pid_list):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": "|".join(pid_list),
        "props": "descriptions",
        "languages": "en",
        "format": "json"
    }
    resp = requests.get(url, params=params, headers=headers, timeout=30)
    return resp.json()

pids = result_df['Qualifier_ID'].tolist()
batches = [pids[i:i+50] for i in range(0, len(pids), 50)]
desc_map = {}

for batch in batches:
    data = get_descriptions_batch(batch)
    if data and 'entities' in data:
        for pid, entity in data['entities'].items():
            desc = entity.get('descriptions', {}).get('en', {}).get('value', '')
            desc_map[pid] = desc
    time.sleep(0.3)

result_df['Description'] = result_df['Qualifier_ID'].map(desc_map).fillna('')
result_df.to_excel("15000politician_qualifier_list.xlsx", index=False)
print("Saved to 15000politician_qualifier_list.xlsx")

Saved to 15000politician_qualifier_list.xlsx
